# 🔢 MNIST Handwritten Digit — CNN Final Version
## ~99.5% Test Accuracy | Real Image Prediction Working
---

In [ ]:
# ─── Step 1: Libraries ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, io
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
tf.random.set_seed(42)

print('='*50)
print('TensorFlow :', tf.__version__)
print('GPU        :', len(tf.config.list_physical_devices('GPU')) > 0)
print('All libraries ready!')
print('='*50)

In [ ]:
# ─── Step 2: Dataset Load ─────────────────────────────────
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(f'Train: {x_train.shape}  |  Test: {x_test.shape}')

In [ ]:
# ─── Step 3: Sample Images ────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('MNIST Sample Digits', fontsize=14, fontweight='bold')
for d in range(10):
    for row in range(2):
        idx = np.where(y_train == d)[0][row]
        axes[row, d].imshow(x_train[idx], cmap='gray')
        axes[row, d].set_title(f'Digit {d}', fontsize=8)
        axes[row, d].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Step 4: Preprocessing ────────────────────────────────
x_tr_n = x_train.astype('float32') / 255.0
x_te_n = x_test.astype('float32')  / 255.0

# CNN shape: (N, 28, 28, 1)
x_tr_cnn = x_tr_n[..., np.newaxis]
x_te_cnn = x_te_n[..., np.newaxis]

y_tr_ohe = to_categorical(y_train, 10)
y_te_ohe = to_categorical(y_test,  10)

cut = int(0.9 * len(x_tr_cnn))
x_val, y_val = x_tr_cnn[cut:], y_tr_ohe[cut:]
x_tr,  y_tr  = x_tr_cnn[:cut], y_tr_ohe[:cut]

print('Preprocessing done!')
print(f'Train:{x_tr.shape}  Val:{x_val.shape}  Test:{x_te_cnn.shape}')

In [ ]:
# ─── Step 5: Data Augmentation ────────────────────────────
# Haath se likhe digits thode tilte/shifted hote hain
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10
)
datagen.fit(x_tr)
print('Data Augmentation ready!')

In [ ]:
# ─── Step 6: CNN Model ────────────────────────────────────
model = keras.Sequential([
    # Block 1
    layers.Conv2D(32, 3, padding='same', input_shape=(28,28,1)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.Conv2D(32, 3, padding='same'),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, 3, padding='same'),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.Conv2D(64, 3, padding='same'),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, 3, padding='same'),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.Dropout(0.25),

    # Classifier
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(), layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ─── Step 7: Train ────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                     patience=3, min_lr=1e-6, verbose=1)
]

print('Training shuru...')
history = model.fit(
    datagen.flow(x_tr, y_tr, batch_size=64),
    epochs=30,
    validation_data=(x_val, y_val),
    callbacks=callbacks, verbose=1
)
print('Training complete!')

In [ ]:
# ─── Step 8: Training Graphs ──────────────────────────────
ep = range(1, len(history.history['accuracy'])+1)
fig, (a1,a2) = plt.subplots(1,2,figsize=(14,5))
a1.plot(ep, history.history['accuracy'],     'b-',  lw=2, label='Train')
a1.plot(ep, history.history['val_accuracy'], 'r--', lw=2, label='Validation')
a1.set_title('Accuracy', fontweight='bold'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(ep, history.history['loss'],     'b-',  lw=2, label='Train')
a2.plot(ep, history.history['val_loss'], 'r--', lw=2, label='Validation')
a2.set_title('Loss', fontweight='bold'); a2.legend(); a2.grid(alpha=0.3)
plt.suptitle('CNN Training Performance', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Step 9: Final Accuracy ───────────────────────────────
loss, acc = model.evaluate(x_te_cnn, y_te_ohe, verbose=0)
print('='*45)
print(f'  Test Accuracy : {acc*100:.2f}%')
print(f'  Test Loss     : {loss:.4f}')
print(f'  Error Rate    : {(1-acc)*100:.2f}%')
print('='*45)

In [ ]:
# ─── Step 10: Confusion Matrix ────────────────────────────
y_pred = np.argmax(model.predict(x_te_cnn, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred))

In [ ]:
# ─── Step 11: Model Save ──────────────────────────────────
model.save('mnist_cnn_model.h5')
print('Model saved: mnist_cnn_model.h5')

---
## 🖼️ Step 12: APNI IMAGE UPLOAD KARO
### Haath se likhi digit do — CNN accurately predict karega!

**Best tarika:**
- MS Paint kholو → canvas kala karo → safed brush se digit likho → PNG save karo
- Ya safed kagaz pe kala pen se likho → seedhi photo lo

In [ ]:
from google.colab import files
from PIL import Image, ImageEnhance, ImageFilter
import io, numpy as np, matplotlib.pyplot as plt

def predict_my_digit(model):
    """
    Apni digit image upload karo.
    CNN — jo training mein ~99.5% accurate hai — predict karega.
    """
    print('='*50)
    print('  DIGIT PREDICTOR — CNN')
    print('='*50)
    print('Digit image upload karo (JPG/PNG)...')
    uploaded = files.upload()

    for filename, data in uploaded.items():
        img_orig = Image.open(io.BytesIO(data))
        original_display = np.array(img_orig.convert('L'))

        # ── Preprocessing chain ──────────────────────────
        img = img_orig.convert('L')

        # Sharpen karo
        img = img.filter(ImageFilter.SHARPEN)

        # Contrast 3x boost
        img = ImageEnhance.Contrast(img).enhance(3.0)

        arr = np.array(img).astype('float32')

        # White bg → invert (MNIST = black bg, white digit)
        if arr.mean() > 127:
            arr = 255.0 - arr

        # Hard threshold — sirf digit pixels rakho
        arr[arr < 50] = 0

        # Digit bounding box crop
        rows = np.any(arr > 0, axis=1)
        cols = np.any(arr > 0, axis=0)
        if rows.any() and cols.any():
            r0,r1 = np.where(rows)[0][[0,-1]]
            c0,c1 = np.where(cols)[0][[0,-1]]
            arr = arr[r0:r1+1, c0:c1+1]

        # Thoda padding — crop ke baad border rakho
        pad = max(arr.shape) // 6
        arr = np.pad(arr, pad, mode='constant', constant_values=0)

        # 20x20 → center paste in 28x28
        pil20   = Image.fromarray(arr.astype(np.uint8)).resize((20,20), Image.LANCZOS)
        canvas  = Image.new('L', (28,28), 0)
        canvas.paste(pil20, (4,4))
        final   = np.array(canvas).astype('float32') / 255.0

        # CNN input shape: (1, 28, 28, 1)
        inp   = final.reshape(1,28,28,1)
        probs = model.predict(inp, verbose=0)[0]
        pred  = int(np.argmax(probs))
        conf  = float(probs[pred]) * 100

        # ── Display ──────────────────────────────────────
        fig, axes = plt.subplots(1, 3, figsize=(14,4))

        axes[0].imshow(original_display, cmap='gray')
        axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(final, cmap='gray')
        axes[1].set_title('Processed — 28x28 MNIST Style', fontsize=12, fontweight='bold')
        axes[1].axis('off')

        colors = ['#d32f2f' if i==pred else '#90caf9' for i in range(10)]
        axes[2].bar(range(10), probs*100, color=colors, edgecolor='black')
        for i,v in enumerate(probs):
            if v > 0.01:
                axes[2].text(i, v*100+0.5, f'{v*100:.1f}%',
                             ha='center', fontsize=8,
                             fontweight='bold' if i==pred else 'normal')
        axes[2].set_title('Digit Probabilities', fontsize=12, fontweight='bold')
        axes[2].set_xticks(range(10))
        axes[2].set_xlabel('Digit (0–9)')
        axes[2].set_ylabel('Confidence (%)')
        axes[2].set_ylim([0, 115])
        axes[2].grid(axis='y', alpha=0.3)

        plt.suptitle(
            f'Predicted Digit: {pred}   |   Confidence: {conf:.1f}%',
            fontsize=16, fontweight='bold', color='darkgreen')
        plt.tight_layout()
        plt.show()

        print('='*45)
        print(f'  Digit      : {pred}')
        print(f'  Confidence : {conf:.2f}%')
        print(f'  File       : {filename}')
        print('  Top 3:')
        for rank, i in enumerate(np.argsort(probs)[::-1][:3], 1):
            bar = '█' * int(probs[i]*20)
            print(f'    {rank}. Digit {i}  {bar}  {probs[i]*100:.2f}%')
        print('='*45)

# Run karo!
predict_my_digit(model)